In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

In [ ]:
# YOUR OPTIMAL PARAMETER (fill this in after Day 5 optimization)

optimal_amp = 0.7  # UPDATE: Your chosen parameter from Day 5 analysis

# DAYS TO COMPARE (Days 1, 5, 10, 15, 16)

days_config = {
    'Day_1': {
        'session_name': 'SK4ASK4BSK2ASK2B_USVBehaviorExperiment_Day1Session1_MMDDYYYY_YO',  # UPDATE
        'csv_file': r'C:\Users\User\usv_detection\USV_DATA\40kHz\Day1_USV.csv',  # UPDATE
        'total_detected': 0,       # Fill after running detection
        'validated_real': 0,       # Fill after full validation
        'validated_false': 0,      # Fill after full validation
        'color': '#9b59b6'         # Purple
    },
    'Day_5': {
        'session_name': 'SK4ASK4BSK2ASK2B_USVBehaviorExperiment_Day5Session1_02032026_YO',
        'csv_file': r'C:\Users\User\usv_detection\USV_DATA\40kHz\SK4ASK4BSK2ASK2B_Day5_USV.csv',  # UPDATE
        'total_detected': 0,       # Fill after running detection
        'validated_real': 0,       # Fill after full validation
        'validated_false': 0,
        'color': '#2ecc71'         # Green
    },
    'Day_10': {
        'session_name': 'SK4ASK4BSK2ASK2B_USVBehaviorExperiment_Day10Session1_MMDDYYYY_YO',  # UPDATE
        'csv_file': r'C:\Users\User\usv_detection\USV_DATA\40kHz\Day10_USV.csv',  # UPDATE
        'total_detected': 0,
        'validated_real': 0,
        'validated_false': 0,
        'color': '#3498db'         # Blue
    },
    'Day_15': {
        'session_name': 'SK4ASK4BSK2ASK2B_USVBehaviorExperiment_Day15Session1_MMDDYYYY_YO',  # UPDATE
        'csv_file': r'C:\Users\User\usv_detection\USV_DATA\40kHz\Day15_USV.csv',  # UPDATE
        'total_detected': 0,
        'validated_real': 0,
        'validated_false': 0,
        'color': '#f39c12'         # Orange
    },
    'Day_16': {
        'session_name': 'SK4ASK4BSK2ASK2B_USVBehaviorExperiment_Day16Session1_MMDDYYYY_YO',  # UPDATE
        'csv_file': r'C:\Users\User\usv_detection\USV_DATA\40kHz\Day16_USV.csv',  # UPDATE
        'total_detected': 0,
        'validated_real': 0,
        'validated_false': 0,
        'color': '#e74c3c'         # Red
    }
}

In [ ]:
def create_validation_tracker(day_name, csv_file):
    """
    Create a tracker CSV for full validation in Audacity
    """
    if not Path(csv_file).exists():
        print(f"{day_name}: CSV not found - run detection first!")
        return None
    
    df = pd.read_csv(csv_file)
    total_usvs = len(df)
    
    # Create validation tracking sheet
    tracker = df[['label', 'start', 'stop']].copy()
    tracker['validated'] = ''  # Fill this in Audacity: 'Real' or 'False'
    
    output_file = f'validation_tracker_{day_name}.csv'
    tracker.to_csv(output_file, index=False)
    
    print(f"\n{day_name}:")
    print(f"  Total USVs detected: {total_usvs}")
    print(f"  Validation tracker saved: {output_file}")
    print(f"  → Open in Audacity and mark each USV as 'Real' or 'False'")
    
    return total_usvs

# Generate trackers for all days
print("=" * 70)
print(f"VALIDATION TRACKERS - Using amp={optimal_amp}")
print("=" * 70)

for day_name, day_data in days_config.items():
    create_validation_tracker(day_name, day_data['csv_file'])

In [ ]:
def plot_cross_day_comparison(days_config, optimal_amp):
    """
    Compare detection quality across Days 1, 5, 10, 15, 16
    """
    # Extract data
    day_names = list(days_config.keys())
    total_detected = [days_config[d]['total_detected'] for d in day_names]
    validated_real = [days_config[d]['validated_real'] for d in day_names]
    validated_false = [days_config[d]['validated_false'] for d in day_names]
    colors = [days_config[d]['color'] for d in day_names]
    
    # Calculate metrics
    precision = [r/(r+f)*100 if (r+f)>0 else 0 for r, f in zip(validated_real, validated_false)]
    false_positive_rate = [f/(r+f)*100 if (r+f)>0 else 0 for r, f in zip(validated_real, validated_false)]
    
    # Create figure
    fig = plt.figure(figsize=(18, 10))
    
    # Plot 1: Total Real USVs per Day
    ax1 = plt.subplot(2, 3, 1)
    bars = ax1.bar(range(len(day_names)), validated_real, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
    ax1.set_ylabel('Real USVs Detected', fontsize=12, fontweight='bold')
    ax1.set_title(f'Real USVs Across Days (amp={optimal_amp})', fontsize=13, fontweight='bold')
    ax1.set_xticks(range(len(day_names)))
    ax1.set_xticklabels(day_names, rotation=45, ha='right')
    ax1.grid(axis='y', alpha=0.3)
    
    for i, (bar, val) in enumerate(zip(bars, validated_real)):
        height = bar.get_height()
        ax1.text(i, height, f'{int(val)}', ha='center', va='bottom', fontweight='bold', fontsize=10)
    
    # Plot 2: Precision Consistency
    ax2 = plt.subplot(2, 3, 2)
    bars = ax2.bar(range(len(day_names)), precision, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
    ax2.set_ylabel('Precision (%)', fontsize=12, fontweight='bold')
    ax2.set_title('Detection Precision Across Days', fontsize=13, fontweight='bold')
    ax2.set_xticks(range(len(day_names)))
    ax2.set_xticklabels(day_names, rotation=45, ha='right')
    ax2.set_ylim(0, 105)
    ax2.axhline(y=90, color='green', linestyle='--', linewidth=2, alpha=0.5, label='90% target')
    ax2.grid(axis='y', alpha=0.3)
    ax2.legend()
    
    for i, (bar, val) in enumerate(zip(bars, precision)):
        height = bar.get_height()
        ax2.text(i, height, f'{val:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=10)
    
    # Plot 3: Total Detected vs Validated Real
    ax3 = plt.subplot(2, 3, 3)
    x = np.arange(len(day_names))
    width = 0.35
    
    bars1 = ax3.bar(x - width/2, total_detected, width, label='Total Detected', 
                    color='lightblue', alpha=0.7, edgecolor='black')
    bars2 = ax3.bar(x + width/2, validated_real, width, label='Validated Real',
                    color='darkgreen', alpha=0.7, edgecolor='black')
    
    ax3.set_ylabel('Count', fontsize=12, fontweight='bold')
    ax3.set_title('Total Detected vs Real USVs', fontsize=13, fontweight='bold')
    ax3.set_xticks(x)
    ax3.set_xticklabels(day_names, rotation=45, ha='right')
    ax3.legend()
    ax3.grid(axis='y', alpha=0.3)
    
    # Plot 4: False Positive Rate
    ax4 = plt.subplot(2, 3, 4)
    bars = ax4.bar(range(len(day_names)), false_positive_rate, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
    ax4.set_ylabel('False Positive Rate (%)', fontsize=12, fontweight='bold')
    ax4.set_title('False Positive Rate Across Days', fontsize=13, fontweight='bold')
    ax4.set_xticks(range(len(day_names)))
    ax4.set_xticklabels(day_names, rotation=45, ha='right')
    ax4.grid(axis='y', alpha=0.3)
    
    for i, (bar, val) in enumerate(zip(bars, false_positive_rate)):
        height = bar.get_height()
        ax4.text(i, height, f'{val:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=9)
    
    # Plot 5: Progression Over Time
    ax5 = plt.subplot(2, 3, 5)
    days_numeric = [1, 5, 10, 15, 16]
    ax5.plot(days_numeric, validated_real, 'o-', linewidth=2.5, markersize=10, color='#2ecc71', label='Real USVs')
    ax5.set_xlabel('Day', fontsize=12, fontweight='bold')
    ax5.set_ylabel('Real USVs Detected', fontsize=12, fontweight='bold')
    ax5.set_title('USV Detection Across Experiment Timeline', fontsize=13, fontweight='bold')
    ax5.grid(True, alpha=0.3)
    ax5.legend()
    
    # Annotate each point
    for i, (day, count) in enumerate(zip(days_numeric, validated_real)):
        ax5.text(day, count + max(validated_real)*0.02, f'{int(count)}', 
                ha='center', va='bottom', fontweight='bold', fontsize=9)
    
    # Plot 6: Summary Table
    ax6 = plt.subplot(2, 3, 6)
    ax6.axis('tight')
    ax6.axis('off')
    
    table_data = []
    for i, day in enumerate(day_names):
        row = [
            day,
            f"{total_detected[i]:,}",
            f"{validated_real[i]:,}",
            f"{precision[i]:.1f}%",
            f"{false_positive_rate[i]:.1f}%"
        ]
        table_data.append(row)
    
    # Add average row
    avg_precision = np.mean(precision)
    avg_fp_rate = np.mean(false_positive_rate)
    table_data.append([
        "AVERAGE",
        f"{int(np.mean(total_detected)):,}",
        f"{int(np.mean(validated_real)):,}",
        f"{avg_precision:.1f}%",
        f"{avg_fp_rate:.1f}%"
    ])
    
    table = ax6.table(cellText=table_data,
                     colLabels=['Day', 'Total\nDetected', 'Real\nUSVs', 'Precision', 'FP Rate'],
                     cellLoc='center',
                     loc='center',
                     colColours=['lightgray']*5)
    
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 2.2)
    
    # Color code day rows
    for i in range(1, len(day_names) + 1):
        table[(i, 0)].set_facecolor(colors[i-1])
        table[(i, 0)].set_alpha(0.4)
    
    # Highlight average row
    for j in range(5):
        table[(len(day_names) + 1, j)].set_facecolor('yellow')
        table[(len(day_names) + 1, j)].set_alpha(0.3)
    
    ax6.set_title('Summary Statistics', fontsize=13, fontweight='bold', pad=20)
    
    # Overall title
    plt.suptitle(f'Cross-Day USV Detection Analysis (amp={optimal_amp})', 
                 fontsize=17, fontweight='bold', y=0.995)
    plt.tight_layout(rect=[0, 0, 1, 0.985])
    
    # Save
    plt.savefig(f'cross_day_comparison_amp{optimal_amp}.png', dpi=300, bbox_inches='tight')
    print(f"\n✓ Plot saved as: cross_day_comparison_amp{optimal_amp}.png")
    
    plt.show()
    
    # Print summary
    print("\n" + "=" * 80)
    print("CROSS-DAY ANALYSIS SUMMARY")
    print("=" * 80)
    for i, day in enumerate(day_names):
        print(f"\n{day}:")
        print(f"  Total Detected:  {total_detected[i]:,}")
        print(f"  Real USVs:       {validated_real[i]:,}")
        print(f"  Precision:       {precision[i]:.1f}%")
        print(f"  FP Rate:         {false_positive_rate[i]:.1f}%")
    
    print(f"\n{'='*80}")
    print(f"AVERAGES:")
    print(f"  Avg Real USVs:   {int(np.mean(validated_real)):,}")
    print(f"  Avg Precision:   {avg_precision:.1f}%")
    print(f"  Consistency:     {np.std(precision):.1f}% std dev")

In [ ]:
# RUN THIS AFTER you've:
# 1. Run detection on all 5 days with your optimal parameter
# 2. Fully validated all USVs in Audacity for each day
# 3. Updated the numbers in Cell 2

plot_cross_day_comparison(days_config, optimal_amp)